> ## SOLUTION / ANSWER KEY &mdash; Lab 4.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-feature-extraction-head.ipynb`](../lab-07-feature-extraction-head.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 4.7 &mdash; Transfer Learning: Frozen Real Features + a Trainable Head

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Turn text into real feature vectors with a frozen transformer
- Train only a small classifier head on those features
- See why this is fast, data-efficient transfer learning

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Transfer learning](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-04-07")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
**Transfer learning** in its cheapest form: keep a powerful **frozen** transformer as a feature
extractor and train only a small **head**. Here `extract_features()` runs the **real** bert-tiny under
`torch.no_grad()` (no weight updates) and mean-pools its token vectors; `LogisticRegression` is the
head. The full fine-tune (labs 4.10&ndash;4.12) *unfreezes* the backbone &mdash; this is the same
shape, one step cheaper.

## Build it
`extract_features()` (given) is the real frozen backbone. Transform both splits and train the head.

In [ ]:
import numpy as np
_FE = {}
def extract_features(texts):
    """REAL frozen features: forward pass through prajjwal1/bert-tiny, mean-pool over real tokens."""
    import torch
    from transformers import AutoTokenizer, AutoModel
    if not _FE:
        name = "prajjwal1/bert-tiny"
        _FE["tok"] = AutoTokenizer.from_pretrained(name)
        _FE["mdl"] = AutoModel.from_pretrained(name); _FE["mdl"].eval()
    if isinstance(texts, str): texts = [texts]
    enc = _FE["tok"](texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad(): out = _FE["mdl"](**enc)          # frozen: no gradients, no weight updates
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)   # mean over REAL tokens only
    return pooled.numpy()

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def run(Xtr_text, Xval_text, ytr, yval):
    Xtr = extract_features(Xtr_text)
    Xval = extract_features(Xval_text)
    head = LogisticRegression(max_iter=1000)
    head.fit(Xtr, ytr)
    return Xtr.shape, accuracy_score(yval, head.predict(Xval))

## Run it for real
Extract real features, train the head, and score on held-out data.

In [ ]:
try:
    # A tiny labelled sentiment dataset (1 = positive, 0 = negative)
    SENT = [
        ("i love this it is great", 1), ("a great and wonderful film", 1),
        ("truly wonderful i love it", 1), ("excellent and brilliant work", 1),
        ("the best most brilliant story", 1), ("i love how great it is", 1),
        ("wonderful excellent and great fun", 1), ("a brilliant and great success", 1),
        ("great fun i really love it", 1), ("the best film wonderful and brilliant", 1),
        ("excellent great and lovely work", 1), ("i love this brilliant great film", 1),
        ("wonderful great and the best", 1), ("so good i love it great", 1),
        ("i hate this it is terrible", 0), ("a terrible and awful film", 0),
        ("truly awful i hate it", 0), ("boring and terrible work", 0),
        ("the worst most boring story", 0), ("i hate how bad it is", 0),
        ("awful boring and dull mess", 0), ("a terrible and bad failure", 0),
        ("boring mess i really hate it", 0), ("the worst film awful and boring", 0),
        ("terrible bad and dull work", 0), ("i hate this awful boring film", 0),
        ("awful terrible and the worst", 0), ("so bad i hate it terrible", 0),
    ]
    texts  = [t for t, y in SENT]
    labels = [y for t, y in SENT]
    from sklearn.model_selection import train_test_split
    Xtr_t, Xval_t, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)
    shape, acc = run(Xtr_t, Xval_t, ytr, yval)
    print("frozen feature matrix shape:", shape, "(examples, hidden_size)")
    print("val accuracy (frozen features + trained head):", round(acc, 3))
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The features are **128-dimensional real** bert-tiny vectors &mdash; you never touched the transformer's weights (`torch.no_grad`).
- A tiny head trained on a handful of examples generalises, because the **frozen backbone already encodes meaning**.
- This is **cheap and fast**: no backprop through the big model, no GPU. It is the workhorse before you ever unfreeze and fine-tune.

## Your turn (open task &mdash; no grader)
Swap the head for an SVM (`from sklearn.svm import SVC`) or change the backbone in
`extract_features` to `distilbert-base-uncased` &mdash; does accuracy change? Add your own labelled
sentences. A "good" answer: you can explain why frozen-feature transfer learning needs so few labels.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    from sklearn.svm import SVC
    from sklearn.metrics import accuracy_score
    from sklearn.model_selection import train_test_split
    Xtr_t, Xval_t, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)
    Xtr = extract_features(Xtr_t); Xval = extract_features(Xval_t)   # SAME frozen real features
    svm = SVC().fit(Xtr, ytr)                                        # only the head changed
    print("SVM head val accuracy:", round(accuracy_score(yval, svm.predict(Xval)), 3))
    print("Few labels suffice: the frozen backbone already encodes meaning, so the head just draws a boundary.")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
Frozen backbone + trainable head = transfer learning at its cheapest. Next we unfreeze the backbone and **fine-tune the whole model** &mdash; the client's headline.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>